Telco Customer Churn - MLP Model Development

End-to-end workflow for customer churn modeling:
- Load and clean the Telco dataset
- Build professional EDA visuals
- Encode categorical features, scale numeric features, and balance classes with SMOTE
- Train an MLP neural network classifier
- Save all artifacts required by the Streamlit dashboard



In [ ]:
# Core data science stack
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd
import plotly.express as px

from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# Update this path if the CSV is moved.
DATA_PATH = Path(r"C:\Users\arshi\Desktop\telco_customer\WA_Fn-UseC_-Telco-Customer-Churn.csv")
ARTIFACT_DIR = Path.cwd() / "models"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)



## Step 1: Data Cleaning and Preprocessing Setup



In [ ]:
def load_and_clean_telco_data(path: Path) -> pd.DataFrame:
    """Load the raw Telco churn CSV and apply dataset-specific cleaning."""
    df = pd.read_csv(path)

    # TotalCharges contains blank strings for a small number of new customers.
    # Converting with errors='coerce' turns those blanks into NaN; 0 is sensible
    # for zero-tenure customers and keeps the full dataset available for modeling.
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    df["TotalCharges"] = df["TotalCharges"].fillna(0.0)

    return df


df = load_and_clean_telco_data(DATA_PATH)
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
df.head()



In [ ]:
# Drop customerID because it is an identifier, not a predictive feature.
df_model = df.drop(columns=["customerID"]).copy()

TARGET = "Churn"
X = df_model.drop(columns=[TARGET])
y_raw = df_model[TARGET]

# Identify feature groups directly from the training data.
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
binary_features = [col for col in categorical_features if X[col].nunique(dropna=False) == 2]
multi_class_features = [col for col in categorical_features if X[col].nunique(dropna=False) > 2]

# Persisted category ordering keeps label encoding stable in the app.
binary_categories = [sorted(X[col].dropna().unique().tolist()) for col in binary_features]
category_options = {
    col: sorted(X[col].dropna().unique().tolist())
    for col in binary_features + multi_class_features
}
numeric_summary = {
    col: {
        "min": float(X[col].min()),
        "max": float(X[col].max()),
        "median": float(X[col].median()),
    }
    for col in numeric_features
}

print("Numeric features:", numeric_features)
print("Binary label-encoded features:", binary_features)
print("One-hot encoded features:", multi_class_features)



In [ ]:
def make_one_hot_encoder():
    """Support both newer and older scikit-learn OneHotEncoder signatures."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(steps=[("scaler", StandardScaler())]),
            numeric_features,
        ),
        (
            "binary",
            Pipeline(
                steps=[
                    (
                        "label_encoder",
                        OrdinalEncoder(
                            categories=binary_categories,
                            handle_unknown="use_encoded_value",
                            unknown_value=-1,
                        ),
                    )
                ]
            ),
            binary_features,
        ),
        (
            "categorical",
            Pipeline(steps=[("one_hot_encoder", make_one_hot_encoder())]),
            multi_class_features,
        ),
    ],
    remainder="drop",
)

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y_raw)
positive_target_value = int(target_encoder.transform(["Yes"])[0])



## Step 2: Visual EDA



In [ ]:
CHURN_COLORS = {"No": "#10B981", "Yes": "#F97316"}
PLOTLY_TEMPLATE = "plotly_dark"

churn_counts = df["Churn"].value_counts().reset_index()
churn_counts.columns = ["Churn", "Customers"]

fig_churn_pie = px.pie(
    churn_counts,
    names="Churn",
    values="Customers",
    hole=0.55,
    color="Churn",
    color_discrete_map=CHURN_COLORS,
    title="Distribution of Customer Churn",
)
fig_churn_pie.update_traces(textposition="inside", textinfo="percent+label")
fig_churn_pie.update_layout(template=PLOTLY_TEMPLATE, title_x=0.02)
fig_churn_pie.show()



In [ ]:
contract_churn = (
    df.groupby("Contract", as_index=False)["Churn"]
    .apply(lambda s: (s == "Yes").mean() * 100)
    .rename(columns={"Churn": "Churn Rate (%)"})
    .sort_values("Churn Rate (%)", ascending=False)
)

fig_contract = px.bar(
    contract_churn,
    x="Contract",
    y="Churn Rate (%)",
    text="Churn Rate (%)",
    color="Contract",
    color_discrete_sequence=["#F97316", "#2563EB", "#10B981"],
    title="Churn Rate by Contract Type",
)
fig_contract.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig_contract.update_layout(template=PLOTLY_TEMPLATE, showlegend=False, title_x=0.02)
fig_contract.show()



In [ ]:
fig_monthly = px.box(
    df,
    x="Churn",
    y="MonthlyCharges",
    color="Churn",
    color_discrete_map=CHURN_COLORS,
    points="outliers",
    title="Monthly Charges vs Churn",
)
fig_monthly.update_layout(template=PLOTLY_TEMPLATE, title_x=0.02)
fig_monthly.show()



## Step 3: MLP Model Engineering



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print(f"Prepared training matrix: {X_train_prepared.shape}")
print(f"Prepared test matrix: {X_test_prepared.shape}")



In [ ]:
# Apply SMOTE only to the training set to avoid leakage into the test split.
smote = SMOTE(random_state=RANDOM_STATE)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_prepared, y_train)

print("Original class balance:", dict(zip(*np.unique(y_train, return_counts=True))))
print("SMOTE class balance:", dict(zip(*np.unique(y_train_balanced, return_counts=True))))



In [ ]:
mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32, 16),
    activation="relu",
    solver="adam",
    alpha=0.0005,
    batch_size=64,
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    random_state=RANDOM_STATE,
)

mlp_model.fit(X_train_balanced, y_train_balanced)



In [ ]:
y_pred = mlp_model.predict(X_test_prepared)
y_proba = mlp_model.predict_proba(X_test_prepared)[:, list(mlp_model.classes_).index(positive_target_value)]

accuracy = accuracy_score(y_test, y_pred)
auc_roc = roc_auc_score(y_test, y_proba)
report = classification_report(y_test, y_pred, target_names=target_encoder.classes_)

print("Classification Report")
print(report)
print(f"Accuracy: {accuracy:.4f}")
print(f"AUC-ROC: {auc_roc:.4f}")



## Save Model Artifacts for Streamlit



In [ ]:
# Pull fitted components out of the ColumnTransformer so the app can load both
# the full preprocessor and the individual scaler/encoders requested.
fitted_scaler = preprocessor.named_transformers_["numeric"].named_steps["scaler"]
fitted_binary_encoder = preprocessor.named_transformers_["binary"].named_steps["label_encoder"]
fitted_one_hot_encoder = preprocessor.named_transformers_["categorical"].named_steps["one_hot_encoder"]

schema = {
    "target": TARGET,
    "id_column": "customerID",
    "feature_columns": X.columns.tolist(),
    "numeric_features": numeric_features,
    "binary_features": binary_features,
    "multi_class_features": multi_class_features,
    "binary_categories": binary_categories,
    "category_options": category_options,
    "numeric_summary": numeric_summary,
    "target_classes": target_encoder.classes_.tolist(),
    "positive_target_value": positive_target_value,
    "accuracy": float(accuracy),
    "auc_roc": float(auc_roc),
    "classification_report": report,
    "data_path": str(DATA_PATH),
}

joblib.dump(mlp_model, ARTIFACT_DIR / "mlp_churn_model.joblib")
joblib.dump(preprocessor, ARTIFACT_DIR / "preprocessor.joblib")
joblib.dump(fitted_scaler, ARTIFACT_DIR / "scaler.joblib")
joblib.dump(fitted_binary_encoder, ARTIFACT_DIR / "binary_label_encoder.joblib")
joblib.dump(fitted_one_hot_encoder, ARTIFACT_DIR / "one_hot_encoder.joblib")
joblib.dump(target_encoder, ARTIFACT_DIR / "target_encoder.joblib")
joblib.dump(schema, ARTIFACT_DIR / "feature_schema.joblib")

print(f"Saved artifacts to: {ARTIFACT_DIR.resolve()}")
print("Artifacts:")
for artifact in sorted(ARTIFACT_DIR.glob("*.joblib")):
    print(f" - {artifact.name}")
